# 04 — Gold: Dimensões SCD Tipo 2 (Spark SQL)

SCD2 para `DimCustomer` e `DimProduct` usando **`UPDATE`** e **`INSERT INTO`** SQL.

**Padrão:**
1. **Carga inicial** — `ValidFrom = 1900-01-01` via `INSERT INTO SELECT`
2. **Incremental** — detecta mudanças com SQL, expira via `UPDATE`, insere nova via `INSERT INTO SELECT`

Sem DeltaTable API — toda a lógica usa `spark.sql("UPDATE ...")`.

In [1]:
import sys
import os
sys.path.insert(0, os.getcwd())
from utils import get_spark, register_catalog, WAREHOUSE_DIR
from datetime import date

spark = get_spark("NorthwindDW SQL - 04 Gold SCD2")
print("Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/03/29 00:47:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 3.5.0


In [2]:
register_catalog(spark)

Catálogo registrado: {'bronze': 11, 'silver': 0, 'gold': 11}


In [3]:
def process_customers_scd2():
    today = str(date.today())
    count = spark.sql("SELECT COUNT(*) AS n FROM gold.DimCustomer").collect()[0]["n"]

    # PASSO 1: Carga inicial
    if count == 0:
        spark.sql(f"""
            INSERT INTO gold.DimCustomer
            SELECT ABS(HASH(CustomerID, '1900-01-01')) AS CustomerSK,
                   CustomerID, CompanyName, ContactName, ContactTitle,
                   City, Country,
                   CAST('1900-01-01' AS DATE) AS ValidFrom,
                   CAST('9999-12-31' AS DATE) AS ValidTo,
                   true                       AS IsCurrent
            FROM bronze.customers
        """)
        n = spark.sql("SELECT COUNT(*) AS n FROM gold.DimCustomer").collect()[0]["n"]
        print(f"Carga inicial: {n} clientes (ValidFrom = 1900-01-01)")
        return

    # PASSO 2: Detectar mudanças
    changed = spark.sql("""
        SELECT b.CustomerID
        FROM bronze.customers b
        JOIN gold.DimCustomer s ON b.CustomerID = s.CustomerID AND s.IsCurrent = true
        WHERE COALESCE(b.ContactName,  '') <> COALESCE(s.ContactName,  '')
           OR COALESCE(b.ContactTitle, '') <> COALESCE(s.ContactTitle, '')
           OR COALESCE(b.City,         '') <> COALESCE(s.City,         '')
           OR COALESCE(b.Country,      '') <> COALESCE(s.Country,      '')
    """)
    changed_ids = [r.CustomerID for r in changed.collect()]

    if changed_ids:
        ids_lit = ", ".join(f"'{i}'" for i in changed_ids)
        # Expirar versão atual
        spark.sql(f"""
            UPDATE gold.DimCustomer
            SET ValidTo   = DATE_SUB(CAST('{today}' AS DATE), 1),
                IsCurrent = false
            WHERE CustomerID IN ({ids_lit}) AND IsCurrent = true
        """)
        # Inserir nova versão
        spark.sql(f"""
            INSERT INTO gold.DimCustomer
            SELECT ABS(HASH(CustomerID, '{today}')) AS CustomerSK,
                   CustomerID, CompanyName, ContactName, ContactTitle,
                   City, Country,
                   CAST('{today}' AS DATE)          AS ValidFrom,
                   CAST('9999-12-31' AS DATE)        AS ValidTo,
                   true                              AS IsCurrent
            FROM bronze.customers WHERE CustomerID IN ({ids_lit})
        """)
        print(f"Mudanças: {len(changed_ids)} clientes expirados e nova versão inserida")
    else:
        print("Nenhuma mudança detectada em DimCustomer.")

    # PASSO 3: Clientes novos
    spark.sql(f"""
        INSERT INTO gold.DimCustomer
        SELECT ABS(HASH(CustomerID, '{today}')) AS CustomerSK,
               CustomerID, CompanyName, ContactName, ContactTitle,
               City, Country,
               CAST('{today}' AS DATE)          AS ValidFrom,
               CAST('9999-12-31' AS DATE)        AS ValidTo,
               true                              AS IsCurrent
        FROM bronze.customers
        WHERE CustomerID NOT IN (
            SELECT DISTINCT CustomerID FROM gold.DimCustomer WHERE IsCurrent = true
        )
    """)

process_customers_scd2()
total   = spark.sql("SELECT COUNT(*) AS n FROM gold.DimCustomer").collect()[0]["n"]
current = spark.sql("SELECT COUNT(*) AS n FROM gold.DimCustomer WHERE IsCurrent = true").collect()[0]["n"]
print(f"DimCustomer: {total} versões totais, {current} ativas (IsCurrent=true)")

Carga inicial: 91 clientes (ValidFrom = 1900-01-01)


DimCustomer: 91 versões totais, 91 ativas (IsCurrent=true)


In [4]:
def process_products_scd2():
    today = str(date.today())

    spark.sql("""
        CREATE OR REPLACE TEMP VIEW bronze_products_enriched AS
        SELECT p.ProductID, p.ProductName,
               c.CategoryName,
               s.CompanyName AS SupplierCompany,
               p.UnitPrice, p.QuantityPerUnit, p.Discontinued
        FROM bronze.products p
        LEFT JOIN bronze.categories c ON p.CategoryID = c.CategoryID
        LEFT JOIN bronze.suppliers  s ON p.SupplierID = s.SupplierID
    """)

    count = spark.sql("SELECT COUNT(*) AS n FROM gold.DimProduct").collect()[0]["n"]
    if count == 0:
        spark.sql(f"""
            INSERT INTO gold.DimProduct
            SELECT ABS(HASH(ProductID, '1900-01-01')) AS ProductSK,
                   ProductID, ProductName, CategoryName, SupplierCompany,
                   UnitPrice, QuantityPerUnit, Discontinued,
                   CAST('1900-01-01' AS DATE) AS ValidFrom,
                   CAST('9999-12-31' AS DATE) AS ValidTo,
                   true                       AS IsCurrent
            FROM bronze_products_enriched
        """)
        n = spark.sql("SELECT COUNT(*) AS n FROM gold.DimProduct").collect()[0]["n"]
        print(f"Carga inicial: {n} produtos (ValidFrom = 1900-01-01)")
        return

    changed = spark.sql("""
        SELECT b.ProductID
        FROM bronze_products_enriched b
        JOIN gold.DimProduct s ON b.ProductID = s.ProductID AND s.IsCurrent = true
        WHERE COALESCE(b.UnitPrice, 0.0) <> COALESCE(s.UnitPrice, 0.0)
           OR b.Discontinued             <> s.Discontinued
    """)
    changed_ids = [r.ProductID for r in changed.collect()]

    if changed_ids:
        ids_lit = ", ".join(str(i) for i in changed_ids)
        spark.sql(f"""
            UPDATE gold.DimProduct
            SET ValidTo   = DATE_SUB(CAST('{today}' AS DATE), 1),
                IsCurrent = false
            WHERE ProductID IN ({ids_lit}) AND IsCurrent = true
        """)
        spark.sql(f"""
            INSERT INTO gold.DimProduct
            SELECT ABS(HASH(ProductID, '{today}')) AS ProductSK,
                   ProductID, ProductName, CategoryName, SupplierCompany,
                   UnitPrice, QuantityPerUnit, Discontinued,
                   CAST('{today}' AS DATE)          AS ValidFrom,
                   CAST('9999-12-31' AS DATE)        AS ValidTo,
                   true                              AS IsCurrent
            FROM bronze_products_enriched WHERE ProductID IN ({ids_lit})
        """)
        print(f"Mudanças: {len(changed_ids)} produtos com nova versão")
    else:
        print("Nenhuma mudança em DimProduct.")

    spark.sql(f"""
        INSERT INTO gold.DimProduct
        SELECT ABS(HASH(ProductID, '{today}')) AS ProductSK,
               ProductID, ProductName, CategoryName, SupplierCompany,
               UnitPrice, QuantityPerUnit, Discontinued,
               CAST('{today}' AS DATE)          AS ValidFrom,
               CAST('9999-12-31' AS DATE)        AS ValidTo,
               true                              AS IsCurrent
        FROM bronze_products_enriched
        WHERE ProductID NOT IN (
            SELECT DISTINCT ProductID FROM gold.DimProduct WHERE IsCurrent = true
        )
    """)

process_products_scd2()
total   = spark.sql("SELECT COUNT(*) AS n FROM gold.DimProduct").collect()[0]["n"]
current = spark.sql("SELECT COUNT(*) AS n FROM gold.DimProduct WHERE IsCurrent = true").collect()[0]["n"]
print(f"DimProduct: {total} versões totais, {current} ativas (IsCurrent=true)")

Carga inicial: 77 produtos (ValidFrom = 1900-01-01)


DimProduct: 77 versões totais, 77 ativas (IsCurrent=true)


In [5]:
print("=" * 55)
print("LAB: Simulação de mudança de dados em bronze")
print("=" * 55)

print("\nCliente ALFKI — versão atual:")
spark.sql("SELECT * FROM gold.DimCustomer WHERE CustomerID = 'ALFKI'").show()

# Simular mudança via SQL UPDATE
spark.sql("UPDATE bronze.customers SET ContactName = 'Maria Anders (UPDATED)' WHERE CustomerID = 'ALFKI'")
print("\nbronze.customers: ContactName de ALFKI atualizado.")

process_customers_scd2()

print("\nCliente ALFKI — versões SCD2 (esperado: 2 linhas):")
spark.sql("SELECT * FROM gold.DimCustomer WHERE CustomerID = 'ALFKI'").show()

dups = spark.sql("""
    SELECT COUNT(*) AS n FROM (
        SELECT CustomerID, COUNT(*) AS cnt FROM gold.DimCustomer
        WHERE IsCurrent = true GROUP BY CustomerID HAVING cnt > 1
    )
""").collect()[0]["n"]
print(f"Clientes com mais de 1 versão ativa: {dups} (esperado: 0)")

LAB: Simulação de mudança de dados em bronze

Cliente ALFKI — versão atual:


+----------+----------+-------------------+------------+--------------------+------+-------+----------+----------+---------+
|CustomerSK|CustomerID|        CompanyName| ContactName|        ContactTitle|  City|Country| ValidFrom|   ValidTo|IsCurrent|
+----------+----------+-------------------+------------+--------------------+------+-------+----------+----------+---------+
| 672125943|     ALFKI|Alfreds Futterkiste|Maria Anders|Sales Representative|Berlin|Germany|1900-01-01|9999-12-31|     true|
+----------+----------+-------------------+------------+--------------------+------+-------+----------+----------+---------+




bronze.customers: ContactName de ALFKI atualizado.


Mudanças: 1 clientes expirados e nova versão inserida



Cliente ALFKI — versões SCD2 (esperado: 2 linhas):


+----------+----------+-------------------+--------------------+--------------------+------+-------+----------+----------+---------+
|CustomerSK|CustomerID|        CompanyName|         ContactName|        ContactTitle|  City|Country| ValidFrom|   ValidTo|IsCurrent|
+----------+----------+-------------------+--------------------+--------------------+------+-------+----------+----------+---------+
| 672125943|     ALFKI|Alfreds Futterkiste|        Maria Anders|Sales Representative|Berlin|Germany|1900-01-01|2026-03-28|    false|
|2006728623|     ALFKI|Alfreds Futterkiste|Maria Anders (UPD...|Sales Representative|Berlin|Germany|2026-03-29|9999-12-31|     true|
+----------+----------+-------------------+--------------------+--------------------+------+-------+----------+----------+---------+



Clientes com mais de 1 versão ativa: 0 (esperado: 0)


In [6]:
spark.sql("UPDATE bronze.customers SET ContactName = 'Maria Anders' WHERE CustomerID = 'ALFKI'")
print("Reset: bronze.customers restaurado ao estado original.")

Reset: bronze.customers restaurado ao estado original.


In [7]:
print("\nResumo SCD2:")
for t in ["gold.DimCustomer", "gold.DimProduct"]:
    total  = spark.sql(f"SELECT COUNT(*) AS n FROM {t}").collect()[0]["n"]
    active = spark.sql(f"SELECT COUNT(*) AS n FROM {t} WHERE IsCurrent = true").collect()[0]["n"]
    print(f"  {t}: total={total}, ativas={active}, históricas={total-active}")


Resumo SCD2:


  gold.DimCustomer: total=92, ativas=91, históricas=1


  gold.DimProduct: total=77, ativas=77, históricas=0
